# 🎵 NB-18: Audio Classification – Topic Detection from Speech
**Goal:** Convert Claude responses to speech, extract MFCC features, train a CNN classifier on audio topic categories.

**Modality:** Audio Classification | **Model:** CNN on MFCC spectrograms

In [ ]:
!pip install librosa soundfile pyttsx3 torch torchvision -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
import pyttsx3, librosa, numpy as np, os, torch, torch.nn as nn
from torch.utils.data import Dataset as TDS, DataLoader

engine = pyttsx3.init(); engine.setProperty("rate", 150)
os.makedirs("./audio_cls", exist_ok=True)

CATS = {"coding":0, "science":1, "math":2, "philosophy":3, "general":4}
def categorize(text):
    text = text.lower()
    if any(k in text for k in ["code","python","algorithm"]): return 0
    if any(k in text for k in ["physics","quantum","molecule","biology"]): return 1
    if any(k in text for k in ["calculate","equation","theorem"]): return 2
    if any(k in text for k in ["consciousness","meaning","ethics"]): return 3
    return 4

mfccs, labels = [], []
print("Extracting MFCC features from TTS audio...")
for i, d in enumerate(data[:60]):
    txt = d["response"][:150].replace("\n"," ")
    path = f"./audio_cls/{i}.wav"
    engine.save_to_file(txt, path); engine.runAndWait()
    if os.path.exists(path):
        try:
            y, sr = librosa.load(path, sr=22050, duration=5.0)
            mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40, n_fft=512)
            mfcc = librosa.util.fix_length(mfcc, size=128, axis=1)
            mfccs.append(mfcc); labels.append(categorize(d["user"]))
        except: pass

print(f"Got {len(mfccs)} samples | Shape: {mfccs[0].shape}")
X = torch.tensor(np.array(mfccs), dtype=torch.float32).unsqueeze(1)  # [N, 1, 40, 128]
y = torch.tensor(labels, dtype=torch.long)

class AudioCNN(nn.Module):
    def __init__(self, n_classes=5):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d((4,4))
        )
        self.fc = nn.Sequential(nn.Flatten(), nn.Linear(128*16,256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256, n_classes))
    def forward(self, x): return self.fc(self.conv(x))

model = AudioCNN(); device = "cuda" if torch.cuda.is_available() else "cpu"; model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

class AudioDS(TDS):
    def __len__(self): return len(y)
    def __getitem__(self, i): return X[i], y[i]

loader = DataLoader(AudioDS(), batch_size=8, shuffle=True)
for epoch in range(10):
    total, correct = 0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb)
        loss = nn.CrossEntropyLoss()(pred, yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item(); correct += (pred.argmax(1)==yb).sum().item()
    print(f"Epoch {epoch+1} | Loss: {total/len(loader):.4f} | Acc: {correct/len(y)*100:.1f}%")
print("✅ Audio topic classifier trained!")
